# Content-Based Filtering

### Data Loading

In [11]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack

In [12]:
df = pd.read_csv('data/newMusicInfo.csv')
df.head()

,track_id,name,artist,spotify_preview_url,tags,genre,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,TROUVHL128F426C441,Come as You Are,NIRVANA,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,218920,0.508,0.826,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,238640,0.515,0.430,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,311014,0.566,0.429,5,-7.826,1,0.0242,0.715000,0.000014,0.1200,0.173,146.365,4
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,https://p.scdn.co/mp3-preview/90e41778392f27b6...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,265506,0.554,0.490,4,-8.046,1,0.0457,0.016800,0.000534,0.1360,0.513,84.275,4
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,https://p.scdn.co/mp3-preview/7388c425022ced92...,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,223106,0.809,0.720,6,-7.776,1,0.1360,0.033700,0.001410,0.8260,0.832,138.556,4


### Selected only the relevant features

In [13]:
df_selected = df[['track_id', 'name','artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo']]
df_selected.head()

,track_id,name,artist,tags,genre,year,danceability,speechiness,tempo
0,TROUVHL128F426C441,Come as You Are,NIRVANA,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,0.508,0.0400,120.012
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,0.515,0.0369,91.841
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,0.566,0.0242,146.365
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,0.554,0.0457,84.275
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,0.809,0.1360,138.556


### Check data type

In [14]:
df_selected.dtypes

track_id         object
name             object
artist           object
tags             object
genre            object
year              int64
danceability    float64
speechiness     float64
tempo           float64
dtype: object

### Copy and clean

In [15]:
df_selected = df[['track_id', 'name', 'artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo']].copy()
df_selected[['name', 'artist', 'tags', 'genre']] = df_selected[['name', 'artist', 'tags', 'genre']].fillna('')

### Combine text features

In [16]:
df_selected['combined_text'] = df_selected['tags'] + ' ' + df_selected['genre'] + ' ' + df_selected['artist']

### TF-IDF on combined text

In [17]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    analyzer='word', 
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=1, # Keep words that appear in at least 1 document
    stop_words='english'
)

# Transform the combined text column into TF-IDF features
text_matrix = tfidf.fit_transform(df_selected['combined_text'])

### Normalize numeric features

In [18]:
# Initialize and apply MinMaxScaler to scale all numeric columns to [0, 1]
scaler = MinMaxScaler()
# Apply scaling to numerical features
numeric_matrix = scaler.fit_transform(df_selected[['danceability', 'speechiness', 'tempo', 'year']])

### Combine all features

In [19]:
combined_matrix = hstack([text_matrix, numeric_matrix]).tocsr()

### Fit KNN model

In [20]:
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(combined_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

### Build name and index mapping

In [21]:
# lowercase song names in mapping
name_to_index = pd.Series(df_selected.index, index=df_selected['name'].str.lower()).drop_duplicates()

### Recommendation function using name

In [36]:
def recommend_songs_by_name(song_name, n_recommendations=5):
    song_name = song_name.lower().strip()
    if song_name not in name_to_index:
        return f"'{song_name}' not found in the dataset."
    
    idx = name_to_index[song_name]
    distances, indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]  # exclude the input song
    
    input_song = df_selected.iloc[[idx]][['track_id', 'name', 'artist', 'genre', 'tags']]
    recommendations = df_selected.iloc[recommended_indices][['track_id', 'name', 'artist', 'genre', 'tags']]
    
    return pd.concat([input_song, recommendations]).reset_index(drop=True)


### Recommendation function using genre

In [59]:
def recommend_songs_by_genre(song_genre, n_recommendations=5):
    song_genre = song_genre.lower().strip()

    # Check if genre exists
    genre_matches = df_selected[df_selected['genre'].str.lower() == song_genre]
    if genre_matches.empty:
        return f"'{song_genre}' not found in the dataset."

    # Pick the first song from that genre for KNN
    idx = genre_matches.index[0]

    # Run KNN on that song's features
    distances, indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]  # exclude the input song

    # Extract all genre-matched songs (your "inputs")
    input_songs = genre_matches[['track_id', 'name', 'artist', 'genre', 'tags']]

    # Extract recommended songs, excluding any that are already in input_songs
    recommendations = df_selected.iloc[recommended_indices][['track_id', 'name',  'genre']]
    recommendations = recommendations[~recommendations['track_id'].isin(input_songs['track_id'])]

    # Combine input + recommendations
    return pd.concat([input_songs, recommendations]).reset_index(drop=True)


### Recommendation function using tags

In [61]:
def recommend_songs_by_tags(song_tags, n_recommendations=5):
    song_tags = song_tags.lower().strip()

    # Check if genre exists
    tags_matches = df_selected[df_selected['tags'].str.lower() == song_tags]
    if tags_matches.empty:
        return f"'{song_tags}' not found in the dataset."

    # Pick the first song from that genre for KNN
    idx = tags_matches.index[0]

    # Run KNN on that song's features
    distances, indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]  # exclude the input song

    # Extract all genre-matched songs (your "inputs")
    input_songs = tags_matches[['track_id', 'name' ,'tags']]

    # Extract recommended songs, excluding any that are already in input_songs
    recommendations = df_selected.iloc[recommended_indices][['track_id', 'name',  'tags']]
    recommendations = recommendations[~recommendations['track_id'].isin(input_songs['track_id'])]

    # Combine input + recommendations
    return pd.concat([input_songs, recommendations]).reset_index(drop=True)


###  User input

In [56]:
user_input = input("🎵 Enter a song name: ")
recommendations = recommend_songs_by_name(user_input, n_recommendations=5)
print("\n🎶 Recommended Songs:\n", recommendations)


🎶 Recommended Songs:
              track_id          name     artist genre  \
0  TRLNZBD128F935E4D8         Creep  RADIOHEAD   RNB   
1  TRIOUGL128F9310CA8        Lurgee  RADIOHEAD  ROCK   
2  TRINQRE128F42BBAA2    Supersonic      OASIS   RNB   
3  TRAFMBS128F423B54E   On Your Own       BLUR   RNB   
4  TRSEHKD128E0782311   Coffee & TV       BLUR  ROCK   
5  TRHSVNJ128E0780D7B  Pure Morning    PLACEBO  ROCK   

                                                tags  
0  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  
1  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  
2  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  
3  INDIE, ALTERNATIVE_ROCK, INDIE_ROCK, BRITISH, ...  
4  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  
5  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  


In [60]:
user_input = input("🎵 Enter a song genre: ")
recommendations = recommend_songs_by_genre(user_input, n_recommendations=5)
print("\n🎶 Recommended Songs:\n", recommendations)


🎶 Recommended Songs:
                track_id                  name      artist genre  \
0    TROUVHL128F426C441       Come as You Are     NIRVANA   RNB   
1    TRLNZBD128F935E4D8                 Creep   RADIOHEAD   RNB   
2    TRYEZGE128F92D04C8        Feel Good Inc.    GORILLAZ   RNB   
3    TRYJPIH128F4262602             This Love    MAROON 5   RNB   
4    TRRIGXH12903CB17E1            The Reason  HOOBASTANK   RNB   
..                  ...                   ...         ...   ...   
690  TRADPIA128E078EE1B      Heart-Shaped Box         NaN  ROCK   
691  TRDMUWU128E078EDDB                  Dumb         NaN  ROCK   
692  TRHMVGF128E078ED9B               Rape Me         NaN  ROCK   
693  TRLZYTY128F931E018                  Blew         NaN  ROCK   
694  TRBNGSV128E078EE1A  Scentless Apprentice         NaN  ROCK   

                                                  tags  
0     ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE  
1    ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  
2

In [62]:
user_input = input("🎵 Enter song genres separated by commas (e.g., Rock, Alternative, Indie): ")
recommendations = recommend_songs_by_tags(user_input, n_recommendations=5)
print("\n🎶 Recommended Songs:\n", recommendations)


🎶 Recommended Songs:
               track_id                         name  \
0   TRDNMMN128F1489831               Hot Soft Light   
1   TRJJCFM128F42B8324                  Poor Doggie   
2   TRGAEDQ12903CB8035      A Ghost Before The Wall   
3   TRXUCNB12903CFF5FB                   On The Bus   
4   TROQFQI128F93239D5              Learn My Lesson   
5   TRUBSXM128F427FEB7                Never Get Old   
6   TRFNODD128F932D739        Semi Precious Weapons   
7   TRYGSXG128F92EC658                 Blowin' Cool   
8   TRKUPQL128E0781D8C                One Way To Go   
9   TROSVGN128F149122B                   Time Thief   
10  TRMLVKW128F423F08D       Death Will Be My Bride   
11  TRJONXS128F9314931                    Englefjes   
12  TRRLBWO128F4239387               Losing Control   
13  TRQNCRJ128F4294EAD       I'm Taking You With Me   
14  TRFQIOB128F92F375F                Gone Too Soon   
15  TREIXQQ128F4277A1B                 Wake The Sun   
16  TRACDGM128E0792ECC              Kids W